In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import pprint

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    use_responses_api=True,
)

pprint.pprint(response.model_dump(), width=500)

{'additional_kwargs': {'refusal': None},
 'content': '응, 잘 지냈어! 😊 너는 어때?',
 'id': 'lc_run--01a0769b-da74-7821-b41e-826bf5a6f343-0',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'finish_reason': 'stop',
                       'id': 'chatcmpl-EL5ubeIYfNOJE2mGyIrUsfwmHuI4f',
                       'logprobs': None,
                       'model_name': 'gpt-5.6-luna',
                       'model_provider': 'openai',
                       'service_tier': 'default',
                       'system_fingerprint': None,
                       'token_usage': {'completion_tokens': 16, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens': 11, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'total_tokens': 27}},
 'tool_calls': [],
 'type': 'ai',
 'usage_metadata': {'input

### ```@tool```: LangChain에서 일반 Python 함수를 LLM이 사용할 수 있는 “도구(Tool)”로 등록하는 데 쓰는 데코레이터
- @tool을 붙이면 get_current_time 함수가 LangChain Tool 객체로 변환되기 때문에 tools나 tool_dict에 넣어서 사용할 수 있음

In [15]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

(복습)
### tools = [get_current_time] → Tool들을 순서대로 모아놓은 리스트라서, 여러 Tool을 LLM에 전달할 때 주로 사용함,
### tool_dict = {"get_current_time": get_current_time} → Tool 이름을 key로 해서 바로 찾을 수 있게 만든 딕셔너리라서, LLM이 요청한 함수 이름으로 실제 함수를 찾아 실행할 때 사용함. 

In [16]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

- Tool이 실제로 등록되어 있다면 LLM은 질문을 보고 알아서 적절한 Tool을 선택할 수 있음
- SystemMessage의 “Tool을 사용할 수 있다”는 문장은 필수라기보다 LLM에게 Tool 사용을 명시적으로 안내하는 추가 지침에 가까움

In [17]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content=[{'id': 'rs_08a435e5b8c138c7006a9d5a2d631c87d08cabfe023071bbb4', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqnVouYpyn95IFW86sADRooljhTzqJeUNw1jG6BNN1gDmHi6JQW-dvLSsrDU4UrrUMm6hm6O5psJ6X6-wJcKHZAESoqf2R5h7qTN1S2iod3zXX785HTj1uY5EJ_EB_69QE2pxdJ3ujWW40YrI3WGS7lhGaBN0FcujQUpq95m6iQtf8dlbaeCPT_WMFW8Q0qi8Es1UVCweKdzyg5OgMDX3YVB7AKJy44K8CQ8yG9UQA5rrmU3_MUq5ofcOs_NWA-ftTkMypgGLxVd09764F9_rcYz-lhfsvRcPtc3WL70mizJJpEThwZkv9Ih02e9ctQTHfE_P2SDxPyrizLQ-tU-gDMmktSBQr6XzySuiU3q82xOSaF5jalQH1nU0AeMf5c2vTtUSBjGrbw7VC59KEIj2WCnVoiGXOWSKA-cNB_N-erNfqy8OS_xPN8OWsG3uyMuCdsrmuXYAtZqwKKRR5jF-ZGqPfBnfrAH1A-hxgCwlt8uY0VL6Ej327_udAEpK3R7-QJS31WMWugkSW0ciIRODl5CjsCgi5P6RMOzRtgj0cZWUUEBaDkCQSR5CfJ918IqebjNPR1jaBduUQ7M2yUi8l0Ww48EOjr1W42vegbQl7HPrLnAn0HEiaZVEHpXr3FPpZoM8yZ

- LLM이 요청한 Tool을 실제로 실행하고 그 결과를 messages에 추가하는 단계

In [29]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

data = response.model_dump()
for item in data["content"]:
    item.pop("encrypted_content", None)

pprint.pprint(data, width=250)

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-09-06 21:26:31 
{'additional_kwargs': {},
 'content': [{'content': [], 'id': 'rs_08a435e5b8c138c7006a9d5a2d631c87d08cabfe023071bbb4', 'summary': [], 'type': 'reasoning'},
             {'arguments': '{"timezone":"Asia/Seoul","location":"부산"}',
              'call_id': 'call_S3rLp7FnDPSzh9Vs4eKgXF1y',
              'id': 'fc_08a435e5b8c138c7006a9d5a2db08087d089e9ff87899043b1',
              'name': 'get_current_time',
              'status': 'completed',
              'type': 'function_call'}],
 'id': 'resp_08a435e5b8c138c7006a9d5a2ce22c87d0b717fed9c06f23e1',
 'invalid_tool_calls': [],
 'name': None,
 'response_metadata': {'created_at': 1788697132.0,
                       'id': 'resp_08a435e5b8c138c7006a9d5a2ce22c87d0b717fed9c06f23e1',
                       'metadata': {},
                       'model': 'gpt-5.6-luna',
                       'model_name': 'gpt-5.6-luna',
                       'model_provider': 'op

- tool_calls → get_current_time 호출 결정 + timezone="Asia/Seoul", location="부산" 전달
- content의 function_call → Responses API가 실제로 요청한 함수 호출 정보 (call_id, arguments, name)
---------------

- Tool 실행 결과까지 포함된 messages를 다시 LLM에게 보내서, 최종 자연어 답변을 받는 단계

In [34]:
llm_with_tools.invoke(messages)

AIMessage(content=[{'id': 'rs_08a435e5b8c138c7006a9d5cadb7bc87d0a21f9ad659d92124', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqnVyvLAPHOW9UNFin-qtNp-GuW59__b0FQNAaSErIMfcMPFaedmsHGSLKtcteE0nUkQnFrOw1sANX6wB-PYqF2rQwoj-ABJYRf83hthSeTad4HGbZmh5cxG_1UNuqZrapIAlRFRgvNfXxYXqyWPsnnnAMB8YbtI7HO5XRPkbe2LJUQnNybnPQLV1DC9ygglDTFoDJfDcbeGnjFMUxrfPpWGY92d_Ep5Y7CDRmk34Erch8uTUJJ6l5UxIsojLOAPBrY-IyJLYzIrGyX0rUvHepzXr8bMUSoLVJ4Si2lOMnHcoyX_5EeTFFRGY_EjD7VoCmvqgNLD3vWDI2UVV-i5NX9mSCsH_lRh7mA_Yutv88u9XF9UiPNmRkbpVqZ06MLAB23h07L8hGeeoHmfjC6Dr8kWd0zhDBPtDJb29kKCZU93TyQPweVfNDYJ_yc_PrNdNqC77QibrjpSAGdyn-ZGKJqZRApoZJQYS1bUDuKAtYFVQGdTBDpT4Uy8nBbhLeSEqR61zvLcjW3nZhoi6hF0fApzFUaV8Wz-Tw1bfy1PMAH2yW_zspvhnLtHqXCrpI_VSe_p3ubnAug2rAvw0xnAvbmMwqgRuo49wSsEIAQUyBNw8t1SAXkucOSZO3FKOBkm9TUKV9WfnF3xFqkiKJZ9x3Gx0ARtm2pNmhp3cedIN1lWnxVh6tO-yvY0HTlXfAaB1ggKxxsfX0Y8Wj9SCfzGjrHVPfuQqsYO06RUTz1JMNWjTONdjpo39LC5_2VNCx-emu3gJJ0P4ay5_PddB4N8ddb3sr0bIOy_U-xbh_K5M0fnZr_CyHdzOvWlZoa75T02njmoH0